In [1]:
import keras
import unicodedata
from keras import layers
from keras import ops

from keras.layers import Layer, Embedding, Input, Dense, GlobalAveragePooling1D, Dropout, MultiHeadAttention, LayerNormalization 
from keras.models import Model

from pathlib import Path
import json
import numpy as np
import tensorflow as tf

2026-05-03 23:09:50.542032: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
SAVE_DIR = Path("/home/valvoja/projects/2026-Spring-Neural-Network-Project/project/models/fi_en_translator")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Introduction  

The purpose of this project is to learn about transformer architecture, specifically its encoder and decoder components, their function and purpose as well as concepts such as cross-attention.  
This project is done with the guidance of [Deep Learning With Python](https://deeplearningwithpython.io/) By François Chollet and Matthew Watson

In [3]:
# Layer and Positional embedding
# Either together or in sequence

# Encoder block starts here

# Input for the encoder block will be the origin language sentence (i.e. Finnish)

# Save original input into a variable to create a residual connection
# Multihead self-attention layer with no mask (except for filling out dud words)

# the output of the attention layer is here added to the original data from the residual connection and then normalised

# Save the outputs of the normalisation layer into another variable for another residual connection
# Output of the normalisation layer is fed into a feedforward layer. (Amount of hidden layers to be adjusted)
# Output of the feedforward layer is added to data from the residual connection and the result normalised


# This process is repeated for as many encoder blocks you want.
# Then the data is ready to be fed into the decoder block

In [4]:
# Layer and Positional embdedding
# Either together or in sequence

# Decoder block starts here

# Input for the decoder block will be the translated language (i.e. English)

# Save original input into a variable to create a residual connection
# Input is fed into a multihead self-attention layer with a causal mask

# Output of the attention layer is added into the data from the residual connection and then normalised

# Save the normalised data into a variable for a residual connection

# The output of the normalisation layer is the query attribute of the layer
# Key and Value attributes are obtained from the final output of the encoder

# Output of this attention layer is added to the data from the residual connection and normalised

# This data is saved to a variable to create a residual connection

# Output of the normalisation layer is fed into a feedforward layer (Amount of hidden layers to be adjusted)
# Output of the feedforward layer is added to the data from the residual connection and normalised

# Output of the normalisation layer is the finished translation

# Data Understanding  

The intiial plan for a dataset was the simply use a substantial text collection (such as an ebook) from both languages. However upon learning more about transformer structure this proved to be a unviable approach.  
For the transformer to properly work the dataset instead needs to be in word pairs, where first of the pair is in the source language i.e. the language to be translated and the other one in the target language i.e. the language to translate to.  
Luckily there exists multiple choices for viable datasets even in this format such as transcripts written down in multiple languages or volunteer collection of translated sentences.  
The latter of the options is the chosen approach for this project.  
Dataset sourced from https://tatoeba.org/en/downloads


## Loading Data

In [5]:
text_file = "/home/valvoja/projects/2026-Spring-Neural-Network-Project/dataset-local/Sentence pairs in Finnish-English - 2026-04-30.tsv"

text_pairs = []
with open(text_file, encoding='utf-8') as f:
    for line in f:
        val1, finnish, val2, english = line.strip().split("\t")
        english = "[start] " + english + " [end]"
        text_pairs.append((finnish, english))
        
print(text_pairs[:10])

[('Lue mittarin lukema.', '[start] Read the meter. [end]'), ('Imuroin pölyt lattialta pölynimurilla.', '[start] I sucked up the dust on the floor with a vacuum cleaner. [end]'), ('Kuulin sen suoraan naapuriltani.', '[start] I heard about it at first hand from my neighbor. [end]'), ('Heinäkuun kymmenentenä avaamme Sapporoon sivuliikkeen.', '[start] On July 10, we will open our Sapporo branch. [end]'), ('Elämänkerran kirjoittamisessa on vaikeaa se, että se on puoliksi dokumentointia ja puoliksi taidetta.', '[start] The difficulty with biography is that it is partly record and partly art. [end]'), ('Isäni työskentelee insinöörinä siinä tehtaassa.', '[start] My father works at the factory as an engineer. [end]'), ('Se laulaja oli parhaimmillaan tuon laulun aikaan.', '[start] The singer was at his best in that song. [end]'), ('Tuuli puhalsi sateen suoraan kasvoilleni.', '[start] A gust of wind blew a shower of rain directly into my face. [end]'), ('Hän huomasi soutuveneen kaukaisuudessa.', 

## Normalising Text

In [6]:
def normalize(text):
    """Lowercase and normalize unicode (e.g. accents)."""
    text = text.lower().strip()
    text = unicodedata.normalize("NFC", text)
    return text

text_pairs = [(normalize(fi), normalize(en)) for fi, en in text_pairs]

In [7]:
print(text_pairs[:10])
print(text_pairs[0][0])
print(text_pairs[0][1])

[('lue mittarin lukema.', '[start] read the meter. [end]'), ('imuroin pölyt lattialta pölynimurilla.', '[start] i sucked up the dust on the floor with a vacuum cleaner. [end]'), ('kuulin sen suoraan naapuriltani.', '[start] i heard about it at first hand from my neighbor. [end]'), ('heinäkuun kymmenentenä avaamme sapporoon sivuliikkeen.', '[start] on july 10, we will open our sapporo branch. [end]'), ('elämänkerran kirjoittamisessa on vaikeaa se, että se on puoliksi dokumentointia ja puoliksi taidetta.', '[start] the difficulty with biography is that it is partly record and partly art. [end]'), ('isäni työskentelee insinöörinä siinä tehtaassa.', '[start] my father works at the factory as an engineer. [end]'), ('se laulaja oli parhaimmillaan tuon laulun aikaan.', '[start] the singer was at his best in that song. [end]'), ('tuuli puhalsi sateen suoraan kasvoilleni.', '[start] a gust of wind blew a shower of rain directly into my face. [end]'), ('hän huomasi soutuveneen kaukaisuudessa.', 

## Preparing Data

### Train, Test and Validation Splits

In [8]:
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

In [9]:
batch_size = 32
vocab_size = 10000
sequence_length = 20

## Keras Layers Method Tokenizing

In [10]:
english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
finnish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
)

train_english_texts = [pair[0] for pair in train_pairs]
train_finnish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
finnish_tokenizer.adapt(train_finnish_texts)

I0000 00:00:1777838995.738648    1167 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5562 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070, pci bus id: 0000:01:00.0, compute capability: 8.6


### Remerging Tokenized Texts

In [11]:
def format_dataset(eng, fin):
    eng = english_tokenizer(eng)
    fin = finnish_tokenizer(fin)
    features = {"english": eng, "finnish": fin[:, :-1]}
    labels = fin[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, fin_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    fin_texts = list(fin_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, fin_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

## Token and Position Encoding

In [12]:
class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        # Computes incrementing positions [0, 1, 2...] for each
        # sequence in the batch
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

In [13]:
#hidden_dim = 256
#intermediate_dim = 2056
#num_heads = 8

embed_dim = 32 # dimension of word embeddings (token + pos)
key_dim = 32
dense_dim = 64 # 
num_heads = 2
encoder_layers = 3
decoder_layers = 2

In [14]:
# Layer and Positional embedding
# Either together or in sequence

source = keras.Input(shape=(None,), dtype="int64", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(source)

# Encoder block starts here

# Input for the encoder block will be the origin language sentence.
# Save original input into a variable to create a residual connection
for _ in range(encoder_layers):
    residual = x
    
    # Multihead self-attention layer with no mask (except for filling out dud words)
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)  
    
    # Save the outputs of the normalisation layer into another variable for another residual connection
    # the output of the attention layer is here added to the original data from the residual connection and then normalised
    x = LayerNormalization()(attention_output + residual)
    residual = x
    
    # Output of the normalisation layer is fed into a feedforward layer. (Amount of hidden layers to be adjusted)
    ff_output = Dense(dense_dim, activation="relu") (x)
    ff2_output = Dense(embed_dim)(ff_output)
    
    # Output of the feedforward layer is added to data from the residual connection and the result normalised
    x = LayerNormalization()(ff2_output + residual)
    
encoder_output = x
# This process is repeated for as many encoder blocks you want.
# Then the data is ready to be fed into the decoder block

"""
z = GlobalAveragePooling1D()(z) 
#z = Dropout(0.2)(z) 
encoder_output = Dense(1, activation='sigmoid')(z) 
"""

"\nz = GlobalAveragePooling1D()(z) \n#z = Dropout(0.2)(z) \nencoder_output = Dense(1, activation='sigmoid')(z) \n"

In [15]:
# Layer and Positional embdedding
# Either together or in sequence
target = keras.Input(shape=(None,), dtype="int64", name="finnish")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(target)

# Decoder block starts here

# Input for the decoder block will be the translated language
# Save original input into a variable to create a residual connection
for _ in range(decoder_layers): 
    residual = x
    
    # Input is fed into a multihead self-attention layer with a causal mask
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x, use_causal_mask=True)  
    
    # Output of the attention layer is added into the data from the residual connection and then normalised
    x = LayerNormalization()(attention_output + residual)
    
    # Save the normalised data into a variable for a residual connection
    residual = x
    
    # The output of the normalisation layer is the query attribute of the layer
    # Key and Value attributes are obtained from the final output of the encoder
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, encoder_output, encoder_output)  
    
    # Output of this attention layer is added to the data from the residual connection and normalised
    x = LayerNormalization()(attention_output + residual)
    
    # This data is saved to a variable to create a residual connection
    residual = x
    
    # Output of the normalisation layer is fed into a feedforward layer (Amount of hidden layers to be adjusted)
    ff_output = Dense(512, activation="relu")(x)
    ff2_output = Dense(32)(ff_output)
    
    # Output of the feedforward layer is added to the data from the residual connection and normalised
    x = LayerNormalization()(ff2_output + residual)

# Output of the normalisation layer is the finished translation
#output = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)

In [16]:
transformer.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ english             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 32)  │    320,640 │ english[0][0]     │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 32)  │      8,416 │ positional_embed… │
│ (MultiHeadAttentio… │                   │            │ positional_embed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 32)  │          0 │ multi_head_atten… │
│                     │                   │            │ positional_embed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, None, 32)  │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 64)  │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 32)  │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, None, 32)  │          0 │ dense_1[0][0],    │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 32)  │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 32)  │      8,416 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, None, 32)  │          0 │ multi_head_atten… │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 32)  │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None, 64)  │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None, 32)  │      2,080 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, None, 32)  │          0 │ dense_3[0][0],    │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 32)  │         64 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 32)  │      8,416 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, None, 32)  │          0 │ multi_head_atten… │
│                     │                   │            │ layer_normalizat

 Total params: 1,110,160 (4.23 MB)

 Trainable params: 1,110,160 (4.23 MB)

 Non-trainable params: 0 (0.00 B)

# Tidying Up The Structure  

As far as mapping and figuring out the structure goes the above model works well enough.  
However restructuring the encoder and decoder into a class is not only the standard procedure, but it looks nicer, is more easily legible and makes their stacking more clear and concise.  

To accomplish this we simply define each of the necessary layers of the neural network as attributes of the class and then define those attributes in a seperate call function of the class.

In [17]:
class Encoder(keras.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, dropout=0.1):
        super().__init__()
        key_dim = embed_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.attention_norm = layers.LayerNormalization()
        self.ff1 = layers.Dense(dense_dim, activation='relu')
        self.ff2 = layers.Dense(embed_dim)
        self.ff_norm = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)


    def call(self, source, source_mask, training=False):
        residual = x = source
        mask = source_mask[:, None, :]
        
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = self.dropout1(x, training=training)
        x = x + residual
        x = self.attention_norm(x)
        
        residual = x
        x = self.ff1(x)
        x = self.ff2(x)
        x = self.dropout2(x, training=training)
        x = x + residual
        x = self.ff_norm(x)
        return x
        

In [18]:
class Decoder(keras.Layer):
    def __init__(self):
        pass

    def call(self):
        pass


In [19]:
checkpoint_path = SAVE_DIR / "best_transformer.weights.h5"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
]

In [20]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, callbacks=callbacks, epochs=15, validation_data=val_ds)

Epoch 1/15


2026-05-03 23:10:08.577841: I external/local_xla/xla/service/service.cc:163] XLA service 0x7474e00105d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-05-03 23:10:08.577872: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3070, Compute Capability 8.6
2026-05-03 23:10:08.887669: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-05-03 23:10:09.694286: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-05-03 23:10:10.918875: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-05-03 23:10:11.436968: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out bec

  25/2301 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.0896 - loss: 3.3141

I0000 00:00:1777839031.583525    1253 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1918/2301 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.1828 - loss: 2.1169

2026-05-03 23:10:43.295610: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-05-03 23:10:44.550092: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-05-03 23:10:44.550171: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-05-03 23:10:44.550185: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none

2299/2301 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1916 - loss: 2.0683

2026-05-03 23:11:08.227859: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-05-03 23:11:08.393445: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-05-03 23:11:08.931614: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_63', 12 bytes spill stores, 12 bytes spill loads




Epoch 1: val_loss improved from None to 1.41923, saving model to /home/valvoja/projects/2026-Spring-Neural-Network-Project/project/models/fi_en_translator/best_transformer.weights.h5

Epoch 1: finished saving model to /home/valvoja/projects/2026-Spring-Neural-Network-Project/project/models/fi_en_translator/best_transformer.weights.h5
2301/2301 ━━━━━━━━━━━━━━━━━━━━ 73s 18ms/step - accuracy: 0.2433 - loss: 1.7983 - val_accuracy: 0.3280 - val_loss: 1.4192
Epoch 2/15
2296/2301 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3666 - loss: 1.3886
Epoch 2: val_loss improved from 1.41923 to 1.18761, saving model to /home/valvoja/projects/2026-Spring-Neural-Network-Project/project/models/fi_en_translator/best_transformer.weights.h5

Epoch 2: finished saving model to /home/valvoja/projects/2026-Spring-Neural-Network-Project/project/models/fi_en_translator/best_transformer.weights.h5
2301/2301 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.3956 - loss: 1.3106 - val_accuracy: 0.4261 - val_loss: 1.

In [21]:
# Save current model weights
transformer.save_weights(str(SAVE_DIR / "final_transformer.weights.h5"))

# Save tokenizer vocabularies
source_vocab = english_tokenizer.get_vocabulary()  # Finnish tokenizer in current notebook
target_vocab = finnish_tokenizer.get_vocabulary()  # English tokenizer in current notebook

with open(SAVE_DIR / "source_vocab_finnish.json", "w", encoding="utf-8") as f:
    json.dump(source_vocab, f, ensure_ascii=False, indent=2)

with open(SAVE_DIR / "target_vocab_english.json", "w", encoding="utf-8") as f:
    json.dump(target_vocab, f, ensure_ascii=False, indent=2)

# Save important model settings
translator_config = {
    "vocab_size": vocab_size,
    "sequence_length": sequence_length,
    "embed_dim": embed_dim,
    "dense_dim": dense_dim,
    "num_heads": num_heads,
    "key_dim": key_dim,
    "encoder_layers": encoder_layers,
    "decoder_layers": decoder_layers,
}

with open(SAVE_DIR / "translator_config.json", "w", encoding="utf-8") as f:
    json.dump(translator_config, f, indent=2)

print("Translator saved to:", SAVE_DIR)

Translator saved to: /home/valvoja/projects/2026-Spring-Neural-Network-Project/project/models/fi_en_translator


In [22]:
# These results no longer reflect the changed structure of the Transformer

In [23]:
import random
import numpy as np

fin_vocab = finnish_tokenizer.get_vocabulary()
fin_index_lookup = dict(zip(range(len(fin_vocab)), fin_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "start"
    for i in range(sequence_length):
        tokenized_target_sentence = finnish_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = transformer.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = fin_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "end":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    translation = generate_translation(input_sentence)
    translation = translation.replace("start", "").replace("end", "").strip()
    print(translation)

-
ole piilossa kiven takana.


2026-05-03 23:13:32.483299: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-05-03 23:13:33.211829: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_37', 12 bytes spill stores, 12 bytes spill loads

2026-05-03 23:13:33.363593: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_37', 8 bytes spill stores, 8 bytes spill loads

2026-05-03 23:13:33.604460: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_37', 24 bytes spil

be wrong behind behind
-
ottakaa askel eteenpäin.
take care of your business
-
marita kertoi tommille, ettei hän uskonut astrologiaan.
the [UNK] told tom he didnt think he thought
-
mitä sinä yrität sanoa? että hän on taitelija?
i plan to tell that he is a good party
-
tänään lentoaseman läheisyydessä on selkeää ja lämpötila on 20 astetta.
today [UNK] is a temperature and a temperature is 20 minutes


In [24]:
import numpy as np

# In the current notebook:
# english_tokenizer = source tokenizer = Finnish
# finnish_tokenizer = target tokenizer = English

target_vocab = finnish_tokenizer.get_vocabulary()
target_index_lookup = dict(enumerate(target_vocab))

def generate_translation(input_sentence):
    """
    Translates one Finnish sentence into English using the trained transformer.
    input_sentence should be plain Finnish text.
    """

    # Same normalization as training
    input_sentence = normalize(input_sentence)

    # Tokenize Finnish source sentence
    tokenized_input_sentence = english_tokenizer([input_sentence])

    # Because TextVectorization removes brackets, [start] becomes start
    decoded_tokens = ["start"]

    for i in range(sequence_length):
        decoded_sentence = " ".join(decoded_tokens)

        # Tokenize partially generated English sentence
        tokenized_target_sentence = finnish_tokenizer([decoded_sentence])

        # Remove last token because decoder input predicts the next token
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]

        # Predict next English token
        predictions = transformer.predict(
            [tokenized_input_sentence, tokenized_target_sentence],
            verbose=0
        )

        sampled_token_index = int(np.argmax(predictions[0, i, :]))
        sampled_token = target_index_lookup[sampled_token_index]

        if sampled_token == "end":
            break

        # Skip padding / empty tokens if they appear
        if sampled_token not in ["", "[UNK]"]:
            decoded_tokens.append(sampled_token)

    # Remove start token
    return " ".join(decoded_tokens[1:]).strip()

In [25]:
test_sentence = "minä pidän kahvista."
translation = generate_translation(test_sentence)

print("Finnish:", test_sentence)
print("English:", translation)

Finnish: minä pidän kahvista.
English: i like coffee


In [26]:
from pathlib import Path
import torch
import whisper
from IPython.display import Audio, display

audio_path = Path("/home/valvoja/projects/2026-Spring-Neural-Network-Project/assignment-3/hand-picked-dataset/file_1.mp3")

print("Audio exists:", audio_path.exists())
print("Audio path:", audio_path)

#device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"
print("Device:", device)

Audio exists: True
Audio path: /home/valvoja/projects/2026-Spring-Neural-Network-Project/assignment-3/hand-picked-dataset/file_1.mp3
Device: cpu


In [27]:
whisper_model_size = "medium"

whisper_model = whisper.load_model(
    whisper_model_size,
    device=device
)

print(f"Loaded Whisper model: {whisper_model_size}")

Loaded Whisper model: medium


In [28]:
def transcribe_finnish_audio(audio_path, model, play_audio=True):
    """
    Transcribes Finnish audio into Finnish text using OpenAI Whisper.
    This should be used before the custom Finnish→English translator.
    """

    audio_path = Path(audio_path)

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    if play_audio:
        display(Audio(str(audio_path)))

    result = model.transcribe(
        str(audio_path),
        language="fi",
        task="transcribe",
        fp16=(device == "cuda"),
        verbose=False,
    )

    finnish_text = result["text"].strip()

    print("Finnish transcription:")
    print(finnish_text)

    return finnish_text

In [29]:
finnish_text = transcribe_finnish_audio(
    audio_path,
    whisper_model,
    play_audio=True
)

/home/valvoja/miniconda3/envs/keras/lib/python3.11/site-packages/whisper/transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")
100%|██████████| 932/932 [00:08<00:00, 110.44frames/s]

Finnish transcription:
Nykyiset rajoitustoimittavat luonteelta on pääasiassa hyvin yleisiä ja laajoja fyysisten kontaktien vähentämisen tähtävien rajoituksia.


In [30]:
english_text = generate_translation(finnish_text)

print("Finnish transcription:")
print(finnish_text)

print("\nEnglish translation:")
print(english_text)

Finnish transcription:
Nykyiset rajoitustoimittavat luonteelta on pääasiassa hyvin yleisiä ja laajoja fyysisten kontaktien vähentämisen tähtävien rajoituksia.

English translation:
the of of the of is there the there the and and in


### Unload Whisper Model

In [ ]:
import gc
import torch

try:
    del whisper_model
except NameError:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Old Whisper model unloaded.")